In [1]:
from pathlib import Path
import duckdb

ORG_GLOB = (
    Path("../data/organizations")
    .resolve()
    .as_posix()
    + "/**/*.parquet"
)

print("Organization data path:")
print(ORG_GLOB)

Organization data path:
C:/Users/Gulso/Documents/Gul/CredBlock-FL/data/organizations/**/*.parquet


In [2]:
graph_sizes = duckdb.sql(f"""
    WITH entity_stats AS (
        SELECT
            organization,
            COUNT(*) AS login_records,
            COUNT(DISTINCT "User ID") AS unique_users,
            COUNT(DISTINCT "IP Address") AS unique_ips

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY organization
    ),

    pair_stats AS (
        SELECT
            organization,
            COUNT(*) AS unique_user_ip_pairs

        FROM (
            SELECT DISTINCT
                organization,
                "User ID",
                "IP Address"

            FROM read_parquet(
                '{ORG_GLOB}',
                hive_partitioning = true
            )
        )

        GROUP BY organization
    )

    SELECT
        e.organization,
        e.login_records,
        e.unique_users,
        e.unique_ips,

        e.unique_users + e.unique_ips
            AS potential_nodes,

        p.unique_user_ip_pairs,

        e.login_records - p.unique_user_ip_pairs
            AS repeated_interactions,

        ROUND(
            100.0 * p.unique_user_ip_pairs
            / e.login_records,
            2
        ) AS unique_pair_pct

    FROM entity_stats e

    JOIN pair_stats p
        USING (organization)

    ORDER BY organization
""").df()

display(graph_sizes)

,organization,login_records,unique_users,unique_ips,potential_nodes,unique_user_ip_pairs,repeated_interactions,unique_pair_pct
0,Org1,3080150,791567,733381,1524948,1697608,1382542,55.11
1,Org2,3250845,825185,762214,1587399,1787741,1463104,54.99
2,Org3,3520641,861765,796189,1657954,1889871,1630770,53.68
3,Org4,3611088,895714,821804,1717518,1975768,1635320,54.71
4,Org5,3780641,930625,852146,1782771,2068379,1712262,54.71
